In [13]:
import numpy as np
import pandas as pd
import scipy

class Node():
    def __init__(self, 
                 feature_i_star=None, 
                 th_star=None, 
                 left=None, 
                 right=None, 
                 label=None):
        
        # for decision node
        self.feature_i_star = feature_i_star
        self.th_star = th_star
        self.left = left
        self.right = right
        
        # y_hat for leaf nodes
        self.label = label

class DecisionTreeClassifier():
    def __init__(self):
        self.root = None
        
    def build_tree(self, X, Y):

        if len(np.unique(Y)) == 1:
            return Node(label=Y[0])

        n, m = X.shape

        feature_i_star, th_star = self.get_best_split(X, Y)
        if feature_i_star == -1:
             return Node(label=self.calculate_leaf_label(Y))

        feature_values = X[:, feature_i_star]
        left_mask = feature_values<=th_star
        right_mask = feature_values>th_star
        Yi_left = Y[left_mask]
        Yi_right = Y[right_mask]
        Xi_left = X[left_mask]
        Xi_right = X[right_mask]

        if len(Xi_left) == 0 or len(Xi_right) == 0:
            return Node(label=self.calculate_leaf_label(Y))

        left_subtree = self.build_tree(Xi_left, Yi_left)
        right_subtree = self.build_tree(Xi_right, Yi_right)
        
        return Node(feature_i_star, th_star, 
                    left_subtree, right_subtree)
    
    def get_best_split(self, X, Y):
            i_star, th_star = -1, -1
            max_info_gain = -float("inf")
            
            m = X.shape[-1]
            for feature_i in range(m):
                feature_values = X[:, feature_i]
                
                feature_values_sorted = np.sort(feature_values)
                thresholds = np.unique(feature_values)

                for th in thresholds:
                                          
                        left_mask = feature_values<=th
                        right_mask = feature_values>th
                        Yi_left = Y[left_mask]
                        Yi_right = Y[right_mask]
                        Xi_left = X[left_mask]
                        Xi_right = X[right_mask]

                        if len(Xi_left)>0 and len(Xi_right)>0:
                            curr_info_gain = self.information_gain(Y, Yi_left, Yi_right)

                            if curr_info_gain>max_info_gain:
                                i_star = feature_i
                                th_star = th
                                max_info_gain = curr_info_gain
                            
            return i_star, th_star
    
    def entropy_calc(self, y):
         values, counts = np.unique(y, return_counts=True)
         p = counts / counts.sum()
         return scipy.stats.entropy(p, base=2)

    def information_gain(self, parent, l_child, r_child):
           ita = len(l_child) / len(parent)
           return self.entropy_calc(parent) - ita*self.entropy_calc(l_child) - (1-ita)*self.entropy_calc(r_child)
    
        
    def calculate_leaf_label(self, Y):
        values, counts = np.unique(Y, return_counts=True)
        return values[np.argmax(counts)]

    def fit(self, X, Y):
        self.root = self.build_tree(X, Y)

    def predict(self, X):
        predictions = []
        for x in X:
            y_hat  = self.make_prediction(x, self.root)
            predictions.append(y_hat)

        return predictions

    def make_prediction(self, x, tree):
        
        if tree.label is not None: #Leaf
            return tree.label
        
        feature_val = x[tree.feature_i_star]
        if feature_val<=tree.th_star:
            return self.make_prediction(x, tree.left)
        else:
            return self.make_prediction(x, tree.right)


In [ ]:
# Carrega a base de dados a partir de seu caminho
print("Carregando dados...")
data = np.load(".\data\data.npz")
X_train = data["X_train"]
y_train = data["y_train"]
print("X_train carregado de forma: ", X_train.shape)
print("y_train carregado de forma: ", y_train.shape)
print("Montando modelo...")
X_test = data['X_test']

classifier = DecisionTreeClassifier()
classifier.fit(X_train, y_train)
print("Modelo criado com sucesso! Realizando a predição...")
y_test = classifier.predict(X_test)

<>:3: SyntaxWarning: invalid escape sequence '\d'
<>:3: SyntaxWarning: invalid escape sequence '\d'
C:\Users\lucas\AppData\Local\Temp\ipykernel_22576\823113555.py:3: SyntaxWarning: invalid escape sequence '\d'
  data = np.load(".\data\data.npz")


Carregando dados...
X_train carregado de forma:  (2831, 34)
y_train carregado de forma:  (2831,)
Montando modelo...


In [17]:
num_samples = X_test.shape[0]
submission_df = pd.DataFrame({
    'ID': np.arange(1, num_samples + 1),
    'Prediction': y_test
})

submission_df.to_csv('out/submission.csv', index=False)